Load Data & Basic Cleaning

In [68]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

DATA    = Path('../data')
RESULTS = Path('results')
RESULTS.mkdir(parents=True, exist_ok=True)

train      = pd.read_csv(DATA / 'train_test.csv')
validation = pd.read_csv(DATA / 'validation.csv')
december   = pd.read_csv(DATA / 'december_chart_inputs.csv')

train['date']      = pd.to_datetime(train['date'])
validation['date'] = pd.to_datetime(validation['date'])
december['date']   = pd.to_datetime(december['date'])

print('Train shape     :', train.shape, '|', train['date'].min().date(), '->', train['date'].max().date())
print('Validation shape:', validation.shape)
print('December shape  :', december.shape)

Train shape     : (48000, 14) | 2025-01-01 -> 2025-10-31
Validation shape: (12000, 13)
December shape  : (31, 7)


In [69]:
def basic_clean(df):
    df = df.copy()
    df['weight'] = df['weight'].abs()
    df.loc[df['distance'] <= 0, 'distance'] = np.nan
    return df

train_raw = basic_clean(train)
val_raw   = basic_clean(validation)
dec_raw   = december.copy()

# Flag corrupted training labels using distance-aware MAD on log(rate)
valid_dist_mask = train_raw['distance'] > 0
lr = np.log(train_raw.loc[valid_dist_mask, 'posted_rate'])
ld = np.log(train_raw.loc[valid_dist_mask, 'distance'])
fit = np.polyfit(ld, lr, 2)
r = lr - np.polyval(fit, ld)
med_r = np.median(r)
mad = 1.4826 * np.median(np.abs(r - med_r))

train_raw['is_corrupt'] = False
train_raw.loc[valid_dist_mask, 'is_corrupt'] = np.abs(r - med_r) > 6 * mad

print(f'Corrupted rows flagged in train: {train_raw["is_corrupt"].sum()} ({train_raw["is_corrupt"].mean():.2%})')
print('These corrupted rows will be excluded from model training to prevent distorted leaf splits.')

Corrupted rows flagged in train: 677 (1.41%)
These corrupted rows will be excluded from model training to prevent distorted leaf splits.


Feature Engineering & Leakage-Free Preprocessing

In [70]:
def haversine(lat1, lon1, lat2, lon2):
    R = 3958.8
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1; dlon = lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

def get_days_to_quarter_end(dates):
    q_end_month = ((dates.dt.month - 1) // 3 + 1) * 3
    days_in_month = {3: 31, 6: 30, 9: 30, 12: 31}
    q_end_dates = pd.to_datetime([f'{y}-{m:02d}-{days_in_month[m]:02d}' for y, m in zip(dates.dt.year, q_end_month)])
    return (q_end_dates - dates).dt.days

def fit_preprocessors(tr_df):
    clean_tr = tr_df[~tr_df['is_corrupt']].copy()
    eq_weight = clean_tr.groupby('equipment')['weight'].median()
    global_weight = clean_tr['weight'].median()
    date_market = clean_tr.groupby('date')['market_index'].median()
    month_market = clean_tr.groupby(clean_tr['date'].dt.to_period('M'))['market_index'].median()
    global_market = clean_tr['market_index'].median()
    
    pickup_map = clean_tr.groupby('pickup')['posted_rate'].median()
    delivery_map = clean_tr.groupby('delivery')['posted_rate'].median()
    global_rate = clean_tr['posted_rate'].median()
    
    return {
        'eq_weight': eq_weight, 'global_weight': global_weight,
        'date_market': date_market, 'month_market': month_market, 'global_market': global_market,
        'pickup_map': pickup_map, 'delivery_map': delivery_map, 'global_rate': global_rate
    }

def transform_features(df, prep):
    df = df.copy()
    
    # Impute weight
    mask_w = df['weight'].isna()
    df.loc[mask_w, 'weight'] = df.loc[mask_w, 'equipment'].map(prep['eq_weight']).fillna(prep['global_weight'])
    
    # Impute market_index (same-date -> month -> global)
    if 'market_index' in df.columns:
        mask_m = df['market_index'].isna()
        df.loc[mask_m, 'market_index'] = df.loc[mask_m, 'date'].map(prep['date_market'])
        mask_m2 = df['market_index'].isna()
        df.loc[mask_m2, 'market_index'] = df.loc[mask_m2, 'date'].dt.to_period('M').map(prep['month_market']).fillna(prep['global_market'])
    
    # Geographic features
    df['geo_distance'] = haversine(df['pickup_lat'], df['pickup_lon'], df['delivery_lat'], df['delivery_lon'])
    df['distance_ratio'] = df['distance'] / df['geo_distance'].replace(0, np.nan)
    
    # Load & Equipment
    df['weight_per_mile'] = df['weight'] / df['distance'].replace(0, np.nan)
    df['equipment_code'] = df['equipment'].map({'Dry Van': 0, 'Reefer': 1, 'Flatbed': 2}).fillna(0).astype(int)
    
    # Cyclical Calendar features
    df['days_to_quarter_end'] = get_days_to_quarter_end(df['date'])
    df['day_of_week'] = df['date'].dt.dayofweek
    df['day_of_month'] = df['date'].dt.day
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    
    # City encodings
    df['pickup_rate_enc'] = df['pickup'].map(prep['pickup_map']).fillna(prep['global_rate'])
    df['delivery_rate_enc'] = df['delivery'].map(prep['delivery_map']).fillna(prep['global_rate'])
    
    return df

Multi-Fold Forward-Chaining Evaluation (2-Month Forecasting Horizons)

In [71]:
FEATURES = [
    'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon',
    'distance', 'geo_distance', 'distance_ratio',
    'weight', 'weight_per_mile', 'equipment_code',
    'market_index',
    'days_to_quarter_end', 'day_of_week', 'day_of_month', 'is_weekend',
    'pickup_rate_enc', 'delivery_rate_enc'
]

FOLDS = [
    {'name': 'Fold 1 (Jul-Aug)', 'train_end': '2025-06-30', 'val_start': '2025-07-01', 'val_end': '2025-08-31'},
    {'name': 'Fold 2 (Aug-Sep)', 'train_end': '2025-07-31', 'val_start': '2025-08-01', 'val_end': '2025-09-30'},
    {'name': 'Fold 3 (Sep-Oct)', 'train_end': '2025-08-31', 'val_start': '2025-09-01', 'val_end': '2025-10-31'},
]

fold_results = []

print('=== Forward-Chaining Fold Results (HistGradientBoosting on log target) ===\n')
for f in FOLDS:
    tr_slice = train_raw[train_raw['date'] <= f['train_end']].copy()
    va_slice = train_raw[(train_raw['date'] >= f['val_start']) & (train_raw['date'] <= f['val_end'])].copy()
    
    prep = fit_preprocessors(tr_slice)
    tr_feat = transform_features(tr_slice, prep)
    va_feat = transform_features(va_slice, prep)
    
    # Train on clean rows only
    tr_clean = tr_feat[~tr_feat['is_corrupt']]
    
    model = HistGradientBoostingRegressor(
        max_iter=400, learning_rate=0.05, max_leaf_nodes=63, min_samples_leaf=20, random_state=42
    )
    model.fit(tr_clean[FEATURES], np.log(tr_clean['posted_rate']))
    
    preds = np.exp(model.predict(va_feat[FEATURES]))
    clean_mask = ~va_feat['is_corrupt']
    
    mae_all = mean_absolute_error(va_feat['posted_rate'], preds)
    rmse_all = np.sqrt(mean_squared_error(va_feat['posted_rate'], preds))
    
    mae_clean = mean_absolute_error(va_feat.loc[clean_mask, 'posted_rate'], preds[clean_mask])
    rmse_clean = np.sqrt(mean_squared_error(va_feat.loc[clean_mask, 'posted_rate'], preds[clean_mask]))
    mape_clean = np.mean(np.abs((va_feat.loc[clean_mask, 'posted_rate'] - preds[clean_mask]) / va_feat.loc[clean_mask, 'posted_rate'])) * 100
    
    fold_results.append({
        'Fold': f['name'],
        'All MAE': round(mae_all, 2), 'All RMSE': round(rmse_all, 2),
        'Clean MAE': round(mae_clean, 2), 'Clean RMSE': round(rmse_clean, 2),
        'Clean MAPE (%)': round(mape_clean, 2)
    })
    
    name = f['name']
    print(f"{name}:")
    print(f"  All rows   : MAE = ${mae_all:6.2f} | RMSE = ${rmse_all:6.2f}")
    print(f"  Clean rows : MAE = ${mae_clean:6.2f} | RMSE = ${rmse_clean:6.2f} | MAPE = {mape_clean:5.2f}%")

folds_df = pd.DataFrame(fold_results)
print('\n' + folds_df.to_string(index=False))

=== Forward-Chaining Fold Results (HistGradientBoosting on log target) ===

Fold 1 (Jul-Aug):
  All rows   : MAE = $114.69 | RMSE = $627.38
  Clean rows : MAE = $ 62.85 | RMSE = $ 85.46 | MAPE =  2.66%
Fold 2 (Aug-Sep):
  All rows   : MAE = $129.29 | RMSE = $624.53
  Clean rows : MAE = $ 79.05 | RMSE = $102.99 | MAPE =  3.35%
Fold 3 (Sep-Oct):
  All rows   : MAE = $148.45 | RMSE = $645.31
  Clean rows : MAE = $ 93.12 | RMSE = $122.87 | MAPE =  3.91%

            Fold  All MAE  All RMSE  Clean MAE  Clean RMSE  Clean MAPE (%)
Fold 1 (Jul-Aug)   114.69    627.38      62.85       85.46            2.66
Fold 2 (Aug-Sep)   129.29    624.53      79.05      102.99            3.35
Fold 3 (Sep-Oct)   148.45    645.31      93.12      122.87            3.91


Retrain on Full Dataset & Generate Deliverables

In [72]:
# Fit preprocessors on all clean training data (Jan-Oct 2025)
full_prep = fit_preprocessors(train_raw)
train_full_feat = transform_features(train_raw, full_prep)
train_clean_full = train_full_feat[~train_full_feat['is_corrupt']]

print(f'Training final model on {len(train_clean_full):,} uncorrupted historical loads.')

final_model = HistGradientBoostingRegressor(
    max_iter=500, learning_rate=0.05, max_leaf_nodes=63, min_samples_leaf=20, random_state=42
)
final_model.fit(train_clean_full[FEATURES], np.log(train_clean_full['posted_rate']))
print('Final model fit complete.')

Training final model on 47,323 uncorrupted historical loads.
Final model fit complete.


In [73]:
# Generate output/validation_predictions.csv and root validation_predictions.csv
val_feat = transform_features(val_raw, full_prep)
val_preds = np.exp(final_model.predict(val_feat[FEATURES]))
val_preds = np.clip(val_preds, a_min=1.0, a_max=None)

val_output = pd.DataFrame({
    'load_id': validation['load_id'].values,
    'predicted_rate': np.round(val_preds, 2)
})

assert len(val_output) == 12000
assert val_output['predicted_rate'].isna().sum() == 0
assert (val_output['predicted_rate'] > 0).all()

out_dir = Path('../output')
out_dir.mkdir(exist_ok=True)
val_output.to_csv(out_dir / 'validation_predictions.csv', index=False)
print(f'Successfully wrote validation_predictions.csv ({len(val_output)} rows)')
print(f'Mean: ${val_output["predicted_rate"].mean():.2f} | Range: ${val_output["predicted_rate"].min():.2f} - ${val_output["predicted_rate"].max():.2f}')


Successfully wrote validation_predictions.csv (12000 rows)
Mean: $2321.18 | Range: $192.38 - $6729.49
